# 🔍 Knowledge Base Explorer
Query the MOVE vector knowledge base built from crawled company websites.

In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from backend.rag.vector_store import VectorStore
from backend.rag.embedder import embed_text

vs = VectorStore()
print(f"✅ Knowledge base loaded: {vs.count()} chunks")

## 🔎 Search the Knowledge Base
Change `QUERY` to anything you want to ask about the crawled site.

In [ ]:
QUERY = "what does this company do"
N_RESULTS = 5

query_vec = embed_text(QUERY)
results = vs.query(query_vec, n_results=N_RESULTS)

print(f'Query: "{QUERY}"')
print("─" * 70)

for i, r in enumerate(results, 1):
    url = r['metadata'].get('url', 'unknown')
    score = r['score']
    content = r['content'].strip()
    print(f"\n[{i}] score={score:.3f}")
    print(f"     {url}")
    print(f"     {content[:500]}")
    print()

## 📋 Browse All Unique Pages Crawled

In [ ]:
# Get all unique URLs stored in the knowledge base
all_results = vs.collection.get(include=["metadatas"])
urls = sorted(set(
    m.get("url", "unknown")
    for m in all_results["metadatas"]
))

print(f"{len(urls)} unique pages in knowledge base:\n")
for u in urls:
    print(f"  {u}")

## 🔬 Deep Dive — Read Full Content of a Specific Page

In [ ]:
PAGE_URL = "https://beamdata.ai/"   # ← change to any URL from the list above

page_results = vs.collection.get(
    where={"url": PAGE_URL},
    include=["documents", "metadatas"],
)

chunks = page_results["documents"]
print(f"Found {len(chunks)} chunks for {PAGE_URL}\n")
print("─" * 70)
for i, chunk in enumerate(chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

## 🧪 Multi-Query Comparison
Run several queries at once and compare results side by side.

In [ ]:
QUERIES = [
    "what products or services do they offer",
    "who are their target customers",
    "what is their pricing",
    "who is the team or leadership",
]

for query in QUERIES:
    vec = embed_text(query)
    top = vs.query(vec, n_results=1)[0]
    print(f"Q: {query}")
    print(f"   score={top['score']:.3f} | {top['metadata'].get('url', '')}")
    print(f"   {top['content'].strip()[:250]}")
    print()